# Vaani Pilot v2: Stack Bandlimit Before Codecs

Run this CPU Colab notebook only after the original v1 pilot completed. Manual listening showed that the codec-only v1 conditions sounded too close to the original because they did not include the explicit 300-3400 Hz telephone filter.

This notebook reuses the v1 source, original, and bandwidth-limited archives. It generates only the corrected stacked A-law, mu-law, and GSM-FR conditions and writes a separate `vaani_paired_pilot_v2` result. It does not download Vaani again and does not evaluate models.

In [ ]:
import importlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/call-whisper')
V1_OUTPUT = DRIVE_PROJECT_DIR / 'results/vaani_paired_pilot_v1'
V2_OUTPUT = DRIVE_PROJECT_DIR / 'results/vaani_paired_pilot_v2'
V1_ARCHIVES = V1_OUTPUT / 'archives'
V2_ARCHIVES = V2_OUTPUT / 'archives'
WORK_ROOT = Path('/content/vaani_paired_pilot_v2')
SOURCE_DIR = WORK_ROOT / 'source_audio'
PAIRED_AUDIO_DIR = WORK_ROOT / 'paired_audio'
REPO_DIR = Path('/content/CallWhisper-8k')

if not (V1_OUTPUT / 'validation_summary.json').exists():
    raise FileNotFoundError(f'Completed v1 output not found: {V1_OUTPUT}')
os.chdir('/content')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/anshulLuhsna/CallWhisper-8k.git', str(REPO_DIR),
], check=True)
os.chdir(REPO_DIR)
SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas>=2', 'tqdm>=4.66'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], check=True)
importlib.invalidate_caches()
_paired_telephony = importlib.import_module('callwhisper.datasets.paired_telephony')

for directory in (V2_OUTPUT, V2_ARCHIVES, WORK_ROOT, SOURCE_DIR, PAIRED_AUDIO_DIR):
    directory.mkdir(parents=True, exist_ok=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repository commit:', commit)
print('CallWhisper import:', _paired_telephony.__file__)
print('Reusing v1:', V1_OUTPUT)
print('Writing v2:', V2_OUTPUT)

In [ ]:
import tarfile
from tqdm.auto import tqdm

def copy_with_progress(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + '.part')
    total = source.stat().st_size
    with source.open('rb') as input_handle, temporary.open('wb') as output_handle, tqdm(
        total=total, unit='B', unit_scale=True, desc=f'Copying {destination.name}'
    ) as progress:
        while chunk := input_handle.read(8 * 1024 * 1024):
            output_handle.write(chunk)
            progress.update(len(chunk))
    temporary.replace(destination)

def safe_extract(archive_path, destination):
    root = destination.resolve()
    with tarfile.open(archive_path, 'r:gz') as archive:
        members = archive.getmembers()
        for member in members:
            if member.issym() or member.islnk():
                raise RuntimeError(f'Refusing archive link: {member.name}')
            target = (destination / member.name).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f'Unsafe archive path: {member.name}')
        for member in tqdm(members, desc=f'Restoring {archive_path.name}'):
            archive.extract(member, destination, filter='data')

def archive_directory(source_dir, archive_path, arcname):
    temporary = Path('/content') / f'{archive_path.name}.part'
    if temporary.exists():
        temporary.unlink()
    files = sorted(path for path in source_dir.rglob('*') if path.is_file())
    with tarfile.open(temporary, 'w:gz') as archive:
        for path in tqdm(files, desc=f'Archiving {arcname}'):
            archive.add(path, arcname=str(Path(arcname) / path.relative_to(source_dir)))
    copy_with_progress(temporary, archive_path)
    temporary.unlink()

for required in ('source_audio.tar.gz', 'original.tar.gz', 'bandlimit_8k.tar.gz'):
    path = V1_ARCHIVES / required
    if not path.exists():
        raise FileNotFoundError(path)
    safe_extract(path, WORK_ROOT)
    destination = V2_ARCHIVES / required
    if not destination.exists():
        copy_with_progress(path, destination)
print('Restored source, original, and bandlimit audio.')

In [ ]:
import pandas as pd

STATIC_FILES = (
    'dataset_schema.json', 'pilot_selection_summary.json', 'vaani_pilot_500.csv',
    'source_audio_manifest.csv',
)
for name in STATIC_FILES:
    shutil.copy2(V1_OUTPUT / name, V2_OUTPUT / name)
for condition in ('original', 'bandlimit_8k'):
    shutil.copy2(
        V1_OUTPUT / f'vaani_pilot_500_{condition}.csv',
        V2_OUTPUT / f'vaani_pilot_500_{condition}.csv',
    )

pilot_df = pd.read_csv(V1_OUTPUT / 'vaani_pilot_500.csv')
source_df = pd.read_csv(V1_OUTPUT / 'source_audio_manifest.csv')
source_lookup = source_df.set_index('sample_key').to_dict('index')
assert len(pilot_df) == 500
assert pilot_df['speaker_id'].nunique() == 500
print('Pilot and source manifests ready:', len(pilot_df), 'rows')

In [ ]:
from callwhisper.datasets.paired_telephony import (
    CONDITIONS, ffmpeg_version, probe_audio, sha256_file, transform_audio,
    validate_codec_support,
)

NEW_CONDITIONS = CONDITIONS[2:]
print(ffmpeg_version())
print('Corrected conditions:', CONDITIONS)
print('Codec support:', validate_codec_support(CONDITIONS))

for condition in NEW_CONDITIONS:
    condition_dir = PAIRED_AUDIO_DIR / condition
    condition_dir.mkdir(parents=True, exist_ok=True)
    archive_path = V2_ARCHIVES / f'{condition}.tar.gz'
    if archive_path.exists():
        safe_extract(archive_path, WORK_ROOT)
    rows = []
    for row in tqdm(pilot_df.to_dict('records'), desc=f'Building {condition}'):
        source_path = WORK_ROOT / source_lookup[row['sample_key']]['source_audio_path']
        output_path = condition_dir / f"{row['sample_key']}.wav"
        if output_path.exists():
            source_probe = probe_audio(source_path)
            output_probe = probe_audio(output_path)
            metadata = {
                'condition': condition, 'source_sha256': sha256_file(source_path),
                'output_sha256': sha256_file(output_path),
                'source_duration_s': source_probe['duration_s'],
                'duration_s': output_probe['duration_s'],
                'duration_delta_s': abs(output_probe['duration_s'] - source_probe['duration_s']),
                'sample_rate_hz': output_probe['sample_rate_hz'],
                'channels': output_probe['channels'],
            }
        else:
            metadata = transform_audio(source_path, output_path, condition)
        rows.append({
            **row, **metadata, 'audio_path': str(output_path.relative_to(WORK_ROOT)),
            'dataset_revision': '1bf019521d12d742178acc32bf2a42f81cf7c8ef',
        })
    condition_df = pd.DataFrame(rows)
    condition_df.to_csv(V2_OUTPUT / f'vaani_pilot_500_{condition}.csv', index=False)
    if not archive_path.exists():
        archive_directory(condition_dir, archive_path, f'paired_audio/{condition}')
    print('Completed:', condition, len(condition_df))

In [ ]:
frames = [pd.read_csv(V2_OUTPUT / f'vaani_pilot_500_{name}.csv') for name in CONDITIONS]
paired_df = pd.concat(frames, ignore_index=True)
paired_df.to_csv(V2_OUTPUT / 'vaani_pilot_500_all_conditions.csv', index=False)
counts = paired_df.groupby('sample_key')['condition'].nunique()
duration_tolerance = paired_df['source_duration_s'].mul(0.01).clip(lower=0.05)
validation = {
    'source_rows': 500,
    'paired_rows': int(len(paired_df)),
    'expected_paired_rows': 2500,
    'unique_speakers': int(pilot_df['speaker_id'].nunique()),
    'samples_with_all_conditions': int((counts == len(CONDITIONS)).sum()),
    'sample_rates_hz': sorted(int(value) for value in paired_df['sample_rate_hz'].unique()),
    'channels': sorted(int(value) for value in paired_df['channels'].unique()),
    'max_duration_delta_s': float(paired_df['duration_delta_s'].max()),
    'duration_tolerance_violations': int((paired_df['duration_delta_s'] > duration_tolerance).sum()),
    'missing_output_files': int(sum(
        not (WORK_ROOT / path).exists() for path in paired_df['audio_path']
    )),
    'dataset_revision': '1bf019521d12d742178acc32bf2a42f81cf7c8ef',
    'repo_commit': commit,
    'channel_semantics': 'bandlimit first, then codec for every codec condition',
}
assert validation['paired_rows'] == 2500
assert validation['unique_speakers'] == 500
assert validation['samples_with_all_conditions'] == 500
assert validation['sample_rates_hz'] == [16000]
assert validation['channels'] == [1]
assert validation['duration_tolerance_violations'] == 0
assert validation['missing_output_files'] == 0
(V2_OUTPUT / 'validation_summary.json').write_text(
    json.dumps(validation, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
protocol = json.loads((V1_OUTPUT / 'protocol_config.json').read_text(encoding='utf-8'))
protocol.update({
    'conditions': CONDITIONS, 'repo_commit': commit,
    'channel_semantics': validation['channel_semantics'], 'supersedes': 'vaani_paired_pilot_v1',
})
(V2_OUTPUT / 'protocol_config.json').write_text(
    json.dumps(protocol, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
print(json.dumps(validation, indent=2))
print('Corrected v2 saved under:', V2_OUTPUT)

In [ ]:
from IPython.display import Audio, Markdown, display

for key in pilot_df['sample_key'].head(3):
    display(Markdown(f'### Sample `{key}`'))
    for condition in CONDITIONS:
        print(condition)
        display(Audio(str(PAIRED_AUDIO_DIR / condition / f'{key}.wav')))

## Stop and Listen Again

The three stacked codec conditions should now retain their codec character while also sounding recognizably narrowband like `bandlimit_8k`. Confirm there is no silence, clipping, speed change, or unrelated audio. Do not evaluate models until this listening check passes.